In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow_datasets as tfds   # ← replaces emnist package

print("TensorFlow:", tf.__version__)
print("All imports done ✅")

In [ ]:
(X_mnist_train, y_mnist_train), (X_mnist_test, y_mnist_test) = mnist.load_data()

print("MNIST train:", X_mnist_train.shape)  # (60000, 28, 28)
print("MNIST test: ", X_mnist_test.shape)   # (10000, 28, 28)
print("Labels:     ", np.unique(y_mnist_train))  # [0 1 2 ... 9]

# Visualize a few MNIST digits
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_mnist_train[i], cmap='gray')
    ax.set_title(str(y_mnist_train[i]))
    ax.axis('off')
plt.suptitle('MNIST — digits 0-9', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Load EMNIST letters via tensorflow_datasets
ds_train = tfds.load('emnist/letters', split='train', as_supervised=True)
ds_test  = tfds.load('emnist/letters', split='test',  as_supervised=True)

# Convert tf.Dataset → numpy arrays
def dataset_to_numpy(ds):
    images, labels = [], []
    for img, lbl in tfds.as_numpy(ds):
        images.append(img[:, :, 0])   # shape (28,28,1) → (28,28)
        labels.append(lbl)
    return np.array(images), np.array(labels)

print("Converting training set... (takes ~1 min)")
X_em_train, y_em_train = dataset_to_numpy(ds_train)

print("Converting test set...")
X_em_test, y_em_test = dataset_to_numpy(ds_test)

print("EMNIST train:", X_em_train.shape)
print("EMNIST test: ", X_em_test.shape)
print("Labels range:", y_em_train.min(), "to", y_em_train.max())  # 1 to 26

# Fix orientation (EMNIST images are transposed vs MNIST)
X_em_train = np.transpose(X_em_train, (0, 2, 1))
X_em_test  = np.transpose(X_em_test,  (0, 2, 1))

# Remap labels: 1–26 → 10–35
# So: 0-9 = digits, 10-35 = A-Z
y_em_train = y_em_train + 9
y_em_test  = y_em_test  + 9

print("Remapped labels:", y_em_train.min(), "to", y_em_train.max())

# Visualize a few letters
LETTERS = 'ABCDEFGH'
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_em_train[i], cmap='gray')
    ax.set_title(LETTERS[i])
    ax.axis('off')
plt.suptitle('EMNIST — letters A-Z (via tfds)', fontsize=12)
plt.tight_layout()
plt.show()

print("EMNIST loaded ✅")

In [ ]:
# ── Combine ───────────────────────────────────────────────────
X_train_raw = np.concatenate([X_mnist_train, X_em_train], axis=0)
X_test_raw  = np.concatenate([X_mnist_test,  X_em_test],  axis=0)
y_train     = np.concatenate([y_mnist_train, y_em_train], axis=0)
y_test      = np.concatenate([y_mnist_test,  y_em_test],  axis=0)

print("Combined train:", X_train_raw.shape)  # (184800, 28, 28)
print("Combined test: ", X_test_raw.shape)   # (24800, 28, 28)
print("Total classes: ", len(np.unique(y_train)))  # 36

# ── Normalize: 0-255 → 0.0-1.0 ───────────────────────────────
X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32')  / 255.0

# ── Reshape: add channel dim for Conv2D ───────────────────────
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

print("\nFinal X_train:", X_train.shape)   # (184800, 28, 28, 1)
print("Final X_test: ", X_test.shape)     # (30800, 28, 28, 1)
print("y_train range:", y_train.min(), "to", y_train.max())  # 0 to 35

# Label map for display (36 classes)
LABEL_MAP = {i: str(i) for i in range(10)}
LABEL_MAP.update({i+10: chr(65+i) for i in range(26)})
# {0:'0', 1:'1'..9:'9', 10:'A', 11:'B'...35:'Z'}
print("\nLabel map sample:", {k: LABEL_MAP[k] for k in [0,5,9,10,15,35]})

In [ ]:
model = keras.Sequential([

    # Add explicit Input layer — removes the UserWarning
    keras.Input(shape=(28, 28, 1)),

    # Block 1
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # Block 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # Block 3
    layers.Conv2D(128, (3,3), activation='relu'),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(36, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=10,             # More epochs for harder 36-class problem
    batch_size=128,        # Larger batch for larger dataset
    validation_split=0.1,
    verbose=1
)

print("\nTraining complete ✅")

In [ ]:
model.save('cnn_handwriting_model.keras')
print("Model saved ✅")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy over epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss over epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Verify label map covers your actual range
print("y_test unique labels:", np.unique(y_test))
print("LABEL_MAP keys:", sorted(LABEL_MAP.keys()))

# If any mismatch, rebuild it:
LABEL_MAP = {i: str(i) for i in range(10)}
LABEL_MAP.update({i+10: chr(65+i) for i in range(26)})
print("Label map:", LABEL_MAP)

In [ ]:
actual_labels = sorted(np.unique(y_test))
actual_names  = [LABEL_MAP[l] for l in actual_labels]

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc*100:.2f}%")
print(f"Test loss:     {test_loss:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)

plt.figure(figsize=(16, 14))
cm = confusion_matrix(y_test, y_pred, labels=actual_labels)
sns.heatmap(cm, annot=False, cmap='Blues',
            xticklabels=actual_names,
            yticklabels=actual_names)
plt.title('Confusion Matrix — Combined CNN (digits + letters)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred,
      labels=actual_labels,
      target_names=actual_names,
      zero_division=0))

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
indices = np.random.choice(len(X_test), 16, replace=False)

for i, (ax, idx) in enumerate(zip(axes.flat, indices)):
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    pred   = LABEL_MAP[y_pred[idx]]
    actual = LABEL_MAP[y_test[idx]]
    color  = 'green' if pred == actual else 'red'
    ax.set_title(f"P:{pred} A:{actual}", color=color, fontsize=10)
    ax.axis('off')

plt.suptitle('Predictions — green=correct, red=wrong', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
idx = np.random.randint(len(X_test))
sample = X_test[idx]
actual = LABEL_MAP[y_test[idx]]

proba  = model.predict(sample[np.newaxis, ...], verbose=0)[0]
pred   = LABEL_MAP[np.argmax(proba)]
conf   = proba.max() * 100

plt.figure(figsize=(3, 3))
plt.imshow(sample.reshape(28, 28), cmap='gray')
plt.title(f"Predicted: '{pred}'  Actual: '{actual}'\nConfidence: {conf:.1f}%",
          fontsize=11)
plt.axis('off')
plt.show()

# Top 5 predictions
top5_idx  = proba.argsort()[-5:][::-1]
print("Top 5 predictions:")
for i in top5_idx:
    print(f"  {LABEL_MAP[i]}: {proba[i]*100:.2f}%")